# IM-RoTHP: Intensity-Modulated Rotary Transformer Hawkes Process

Este notebook apresenta e avalia o modelo **IM-RoTHP**, uma inovação arquitetural sobre o RoTHP.

### A Hipótese
A percepção temporal em processos de eventos não é linear. Em momentos de alta intensidade ("bursts"), a resolução temporal necessária é diferente de momentos de baixa intensidade. 
O **IM-RoTHP** propõe modular a frequência das Rotary Embeddings (RoPE) com base na intensidade estimada do processo, realizando um *Time-Rescaling* implícito dentro do mecanismo de atenção.

### O Experimento
Compararemos o **IM-RoTHP** contra o **RoTHP** (Baseline) no dataset `retweet`.
Métricas:
1.  **Log-Likelihood (NLL):** Acurácia geral do modelo.
2.  **RMSE de Tempo:** Erro de previsão do próximo evento.
3.  **Análise de Atenção (Visual):** Verificaremos se o IM-RoPE altera os padrões de atenção em bursts.
4.  **Curva de Intensidade:** Comparação visual da $\lambda(t)$ aprendida.

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

try:
    import google.colab
    if not os.path.exists('/content/ufc-easytpp'):
        !git clone https://github.com/hugoramos/ufc-easytpp.git
    project_root = '/content/ufc-easytpp'
except:
    project_root = os.getcwd()
    if os.path.basename(project_root) == 'notebooks':
        project_root = os.path.dirname(project_root)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Patch para FP16 (Estabilidade)
import easy_tpp.model.torch_model.torch_baselayer as baselayer
def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn
baselayer.attention = attention_fixed

# Importar Modelos
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_imrothp import IMRoTHP

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

### 1. Preparação dos Dados e Config

In [ ]:
class ThinningConfig:
    def __init__(self):
        self.num_sample = 1
        self.num_exp = 500
        self.over_sample_rate = 5.0
        self.patience_counter = 5
        self.num_samples_boundary = 5
        self.dtime_max = 5.0

class ModelConfig:
    def __init__(self, num_types, pad_id):
        self.hidden_size = 64
        self.time_emb_size = 64
        self.num_layers = 2
        self.num_heads = 4
        self.dropout_rate = 0.1
        self.use_ln = True
        self.num_event_types = num_types
        self.num_event_types_pad = num_types + 1
        self.pad_token_id = pad_id
        self.loss_integral_num_sample_per_step = 20
        self.use_mc_samples = False
        self.gpu = 0 if torch.cuda.is_available() else -1
        self.thinning = ThinningConfig()

def collate_fn_opt(batch_list, pad_id, time_scale):
    batch_size = len(batch_list)
    max_len = max(len(x['time_since_start']) for x in batch_list)
    pad_time = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    batch_non_pad_mask = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal_mask_base = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_since_start'])
        ts = torch.tensor(item['time_since_start'], dtype=torch.float64)
        td = torch.tensor(item['time_since_last_event'], dtype=torch.float64)
        ev = torch.tensor(item['type_event'], dtype=torch.long)
        ts = (ts - ts[0]) / time_scale
        td = td / time_scale
        pad_time[i, :l] = ts.float()
        pad_delta[i, :l] = td.float()
        pad_type[i, :l] = ev
        batch_non_pad_mask[i, :l] = 1.0
        mask_i = causal_mask_base.clone()
        mask_i[:, l:] = True
        mask_i[l:, :] = True
        attention_mask[i] = mask_i
    return (pad_time, pad_delta, pad_type, batch_non_pad_mask, attention_mask)

# Dataset
dataset = load_dataset("easytpp/retweet")
train_data = dataset['train']
dev_data = dataset['validation']
test_data = dataset['test']

all_deltas = []
for item in train_data:
    all_deltas.extend([d for d in item['time_since_last_event'] if d > 0])
time_scale = np.mean(all_deltas)

num_types = 3
pad_id = 3
collate = lambda x: collate_fn_opt(x, pad_id, time_scale)
train_loader = DataLoader(train_data, batch_size=2048, shuffle=True, collate_fn=collate)
dev_loader = DataLoader(dev_data, batch_size=256, shuffle=False, collate_fn=collate)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False, collate_fn=collate)

### 2. Treinamento Comparativo
Treinamos ambos os modelos lado a lado.

In [ ]:
def train_model(model_cls, name, epochs=20):
    print(f"\n>>> Treinando {name}...")
    config = ModelConfig(num_types, pad_id)
    model = model_cls(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler('cuda')
    
    history = []
    for epoch in range(1, epochs+1):
        model.train()
        train_loss = 0
        train_num = 0
        for batch in train_loader:
            batch = [t.to(device, non_blocking=True) for t in batch]
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                loss, num = model.loglike_loss(batch)
                loss = loss / (num + 1e-9)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * num
            train_num += num
            
        # Validation
        model.eval()
        val_loss = 0
        val_num = 0
        with torch.no_grad():
            for batch in dev_loader:
                batch = [t.to(device) for t in batch]
                with torch.amp.autocast('cuda'):
                    l, n = model.loglike_loss(batch)
                val_loss += l.item()
                val_num += n
        
        train_nll = train_loss / (train_num + 1e-9)
        val_nll = val_loss / (val_num + 1e-9)
        history.append({'Epoch': epoch, 'Train NLL': train_nll, 'Val NLL': val_nll})
        
        if epoch % 5 == 0:
            print(f"  Ep {epoch}: Train {train_nll:.4f} | Val {val_nll:.4f}")
            
    return model, pd.DataFrame(history)

# Treinar Baseline e Proposta
model_rothp, hist_rothp = train_model(RoTHP, "RoTHP (Baseline)", epochs=30)
model_im, hist_im = train_model(IMRoTHP, "IM-RoTHP (Proposta)", epochs=30)

### 3. Comparação de Performance (Curvas de Aprendizado)
Visualizamos se a modulação da intensidade ajudou na convergência.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_rothp['Epoch'], hist_rothp['Val NLL'], label='RoTHP', linestyle='--')
ax.plot(hist_im['Epoch'], hist_im['Val NLL'], label='IM-RoTHP', linewidth=2)
ax.set_xlabel('Época')
ax.set_ylabel('NLL (Menor é Melhor)')
ax.set_title('Convergência de Validação: Baseline vs Proposta')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Melhor NLL RoTHP: {hist_rothp['Val NLL'].min():.4f}")
print(f"Melhor NLL IM-RoTHP: {hist_im['Val NLL'].min():.4f}")

### 4. Análise Visual de Intensidade
Plotamos a função de intensidade $\lambda(t)$ para uma amostra de teste para ver as diferenças qualitativas.

In [ ]:
def visualize_intensity_comparison(models, sample_idx=0):
    print(f"\nAnalisando Amostra {sample_idx}...")
    sample = [test_data[sample_idx]]
    batch = collate(sample)
    pad_time, pad_delta, pad_type, _, attn = [t.to(device) for t in batch]
    
    t_seq = pad_time[0].cpu().numpy()
    valid_len = (pad_type[0] != pad_id).sum().item()
    t_seq = t_seq[:valid_len]
    target_type = pad_type[0, 1].item()
    
    t_start = 1
    t_end = min(8, valid_len-1)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for i in range(t_start, t_end):
        t_prev = t_seq[i]
        t_next = t_seq[i+1]
        dt_real = t_next - t_prev
        dts_scan = torch.linspace(0, dt_real, 50, device=device).view(1, 1, -1)
        t_curve = t_prev + dts_scan.cpu().numpy().flatten()
        
        for name, model in models.items():
            model.eval()
            with torch.no_grad():
                L = pad_time.shape[1]
                sample_dtimes = torch.zeros(1, L, 50, device=device)
                sample_dtimes[:, i, :] = dts_scan.squeeze()
                
                lambdas = model.compute_intensities_at_sample_times(
                    pad_time, pad_delta, pad_type, sample_dtimes, attention_mask=attn
                )
                l_curve = lambdas[0, i, :, target_type].cpu().numpy()
                
                style = '--' if 'Baseline' in name else '-'
                width = 1.5 if 'Baseline' in name else 2.5
                ax.plot(t_curve, l_curve, linestyle=style, linewidth=width, 
                        label=name if i==t_start else "")

    for t in t_seq[t_start:t_end+1]:
        ax.axvline(x=t, color='black', alpha=0.2, linestyle=':')
    
    ax.set_title(f'Intensity Comparison: RoTHP vs IM-RoTHP (Type {target_type})')
    ax.set_xlabel('Time')
    ax.set_ylabel('Intensity')
    ax.legend()
    plt.show()

models = {'RoTHP (Baseline)': model_rothp, 'IM-RoTHP (Proposta)': model_im}
visualize_intensity_comparison(models, sample_idx=42)

### 5. Inspeção do Parâmetro Alpha (O "Coração" do IM-RoPE)
Verificamos se o modelo realmente aprendeu a usar a modulação. Se `alpha` for diferente de zero, a hipótese foi validada pelo gradiente.

In [ ]:
alpha_val = model_im.rotary_emb.alpha.item()
print(f"Valor aprendido de Alpha: {alpha_val:.6f}")

if abs(alpha_val) > 0.01:
    print("CONCLUSÃO: O modelo aprendeu ativamente a modular o tempo pela intensidade!")
else:
    print("CONCLUSÃO: O modelo preferiu manter o comportamento estático (alpha ~ 0).")